# Popularity Baseline and Group Metrics

This notebook uses a tiny dataset to make every step inspectable. The production code lives in `src/group_movie_recommender/`.

## 1. Import the reusable project functions

The path block works when the notebook is opened either from the repository root or from the `notebooks` directory.

In [ ]:
from pathlib import Path
import sys

import pandas as pd

project_root = Path.cwd()
if not (project_root / "src").exists():
    project_root = project_root.parent
sys.path.insert(0, str(project_root / "src"))

from group_movie_recommender.evaluation.metrics import (
    build_relevant_item_sets,
    evaluate_shared_rankings,
)
from group_movie_recommender.algorithms.popularity import (
    PopularityRecommender,
    recommend_pairs_by_popularity,
)

## 2. Create a warm catalogue

`trainPositiveCount` is calculated only from positive training interactions. A larger count means a more popular movie.

In [ ]:
warm_catalog = pd.DataFrame(
    {
        "movieId": [30, 10, 20, 40, 50],
        "trainPositiveCount": [20, 100, 60, 10, 5],
    }
)

model = PopularityRecommender.from_warm_catalog(warm_catalog)
model.ranked_movie_ids

The expected global order is `10, 20, 30, 40, 50`. This order is identical for every user because popularity is not personalized.

## 3. Remove movies already seen by either member

User 1 has seen movie 10 and user 2 has seen movie 30. Both movies must be removed from their shared list.

In [ ]:
pairs = pd.DataFrame({"userA": [1], "userB": [2]})
seen_items = {1: {10}, 2: {30}}

recommendations = recommend_pairs_by_popularity(
    pairs,
    warm_catalog,
    seen_items,
    k=3,
)
recommendations

The expected shared recommendation is `20, 40, 50`. Movie 20 is the most popular eligible item.

## 4. Evaluate the same list for both members

A test rating of at least four is a relevant movie. Relevance is restricted to the warm catalogue because other movies cannot be recommended by any evaluated method.

In [ ]:
test_positives = pd.DataFrame(
    {
        "userId": [1, 1, 2, 2],
        "movieId": [20, 50, 20, 30],
    }
)
eligible_movie_ids = set(warm_catalog["movieId"])
relevant_items = build_relevant_item_sets(test_positives, eligible_movie_ids)
relevant_items

In [ ]:
pair_metrics, summary = evaluate_shared_rankings(
    recommendations,
    relevant_items,
    catalog_size=len(warm_catalog),
    k=3,
)

display(pair_metrics)
summary

## 5. Interpret the output

- **Recall@3** measures how many relevant movies appear in the list.
- **NDCG@3** also rewards placing relevant movies near the top.
- **Average NDCG@3** measures overall group satisfaction.
- **Minimum NDCG@3** represents the worse-off member.
- **NDCG gap** measures the difference between the two members.
- **Catalogue coverage** measures how much of the available catalogue appears across all recommendation lists.